# Data Mining Assignment — Spark Data Lake Preprocessing

**Dataset:** `sales.csv`  
**Environment:** Google Colab + PySpark  
**Layers:** Bronze (raw, unchanged) → Silver (basic preprocessing)

LCI2023008 - SHREYA JAMBHALE


In [20]:
!pip -q install pyspark


In [21]:
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.window import Window
import os, glob, shutil, re, json

spark = (
    SparkSession.builder
    .appName("DataMining-Spark-DataLake")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)


Spark version: 4.0.4


## 1. Data Lake Structure

In [22]:
BASE = "/content/data_lake"
BRONZE = os.path.join(BASE, "bronze")
SILVER = os.path.join(BASE, "silver", "preprocessed_sales")

os.makedirs(BRONZE, exist_ok=True)
os.makedirs(SILVER, exist_ok=True)

print("Created:")
print(BASE)
print(BRONZE)
print(SILVER)


Created:
/content/data_lake
/content/data_lake/bronze
/content/data_lake/silver/preprocessed_sales


## 2. Load and Inspect Data

In [24]:
import os
from google.colab import files

if os.path.exists("sales.csv"):
    print("sales.csv already exists. Using the existing file.")
else:
    uploaded = files.upload()

    if "sales.csv" not in uploaded:
        raise FileNotFoundError(
            "Please upload the instructor-provided file named exactly 'sales.csv'."
        )

print("Using:", "sales.csv")

sales.csv already exists. Using the existing file.
Using: sales.csv


In [25]:
bronze_path = os.path.join(BRONZE, "sales.csv")

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(bronze_path)
)

print("Columns:")
print(df.columns)

print("\nSchema:")
df.printSchema()

print("\nFirst few rows:")
df.show(5, truncate=False)

print("\nTotal record count:", df.count())


Columns:
['order_id', 'customer_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'discount_percent', 'payment_method', 'city', 'state', 'order_date', 'order_status']

Schema:
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)


First few rows:
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|un

### Data Quality Checks

In [26]:
# Exact duplicate rows
duplicate_count = df.count() - df.dropDuplicates().count()
print("Exact duplicate rows:", duplicate_count)

# Null counts
null_counts = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])
print("\nNull values by column:")
null_counts.show(truncate=False)

# Blank-string counts for text columns
string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, T.StringType)]
if string_cols:
    blank_counts = df.select([
        F.sum(
            F.when(F.trim(F.col(c)) == "", 1).otherwise(0)
        ).alias(c)
        for c in string_cols
    ])
    print("Blank/whitespace-only values in text columns:")
    blank_counts.show(truncate=False)
else:
    print("No string columns detected.")


Exact duplicate rows: 10

Null values by column:
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|0       |0          |10           |0      |0       |0       |0         |0               |8             |8   |0    |0         |0           |
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+

Blank/whitespace-only values in text columns:
+-----------+-------------+-------+--------+--------------+----+-----+----------+------------+
|customer_id|customer_name|product|category|payment_method|city|state|order_date|order_status|
+--------

In [27]:
def normalize_name(name):
    return re.sub(r"[^a-z0-9]", "", name.lower())

def find_column(candidates):
    normalized = {normalize_name(c): c for c in df.columns}
    # exact normalized match
    for candidate in candidates:
        key = normalize_name(candidate)
        if key in normalized:
            return normalized[key]
    # contains match
    for c in df.columns:
        nc = normalize_name(c)
        for candidate in candidates:
            key = normalize_name(candidate)
            if key in nc or nc in key:
                return c
    return None

quantity_col = find_column([
    "quantity", "qty", "quantityordered", "units", "unitsold"
])

unit_price_col = find_column([
    "unitprice", "unit_price", "priceeach", "price", "sellingprice"
])

discount_col = find_column([
    "discountpercentage", "discountpercent", "discount_pct",
    "discount_percentage", "discount", "discountrate"
])

date_col = find_column([
    "date", "orderdate", "salesdate", "transactiondate",
    "invoicedate", "purchasedate"
])

print("Detected columns:")
print("Quantity:", quantity_col)
print("Unit price:", unit_price_col)
print("Discount percentage:", discount_col)
print("Date:", date_col)


Detected columns:
Quantity: quantity
Unit price: unit_price
Discount percentage: discount_percent
Date: order_date


## 3. Basic Preprocessing - Silver Layer

In [28]:
silver_df = df

# 1. Remove exact duplicate rows
silver_df = silver_df.dropDuplicates()

# 2 & 3. Trim text fields and standardize obvious capitalization.
for field in silver_df.schema.fields:
    c = field.name
    if isinstance(field.dataType, T.StringType):
        silver_df = silver_df.withColumn(c, F.trim(F.col(c)))
        nc = normalize_name(c)
        if not any(x in nc for x in ["email", "url", "id", "code"]):
            silver_df = silver_df.withColumn(
                c,
                F.when(F.col(c).isNotNull(), F.initcap(F.col(c))).otherwise(None)
            )

# 4. Handle missing values.
numeric_cols = [
    f.name for f in silver_df.schema.fields
    if isinstance(f.dataType, (T.IntegerType, T.LongType, T.FloatType, T.DoubleType, T.ShortType, T.DecimalType))
]

for c in numeric_cols:
    if c not in [quantity_col, discount_col, unit_price_col]:
        median_row = silver_df.select(F.expr(f"percentile_approx(`{c}`, 0.5)").alias("median")).first()
        median_value = median_row["median"] if median_row else None
        if median_value is not None:
            silver_df = silver_df.fillna({c: median_value})

# 5. Validate quantity
if quantity_col:
    silver_df = silver_df.withColumn(
        quantity_col,
        F.when(F.col(quantity_col).cast("double") >= 0, F.col(quantity_col).cast("double"))
         .otherwise(None)
    )

# Validate unit price
if unit_price_col:
    silver_df = silver_df.withColumn(
        unit_price_col,
        F.when(F.col(unit_price_col).cast("double") >= 0, F.col(unit_price_col).cast("double"))
         .otherwise(None)
    )

# Validate discount percentage: valid range is 0–100.
if discount_col:
    silver_df = silver_df.withColumn(
        discount_col,
        F.when(
            (F.col(discount_col).cast("double") >= 0) &
            (F.col(discount_col).cast("double") <= 100),
            F.col(discount_col).cast("double")
        ).otherwise(None)
    )

# 6. Standardize dates.
if date_col:
    raw_date = F.trim(F.col(date_col).cast("string"))

    parsed_date = F.coalesce(
        F.expr(f"try_to_timestamp(`{date_col}`, 'yyyy-MM-dd')").cast("date"),
        F.expr(f"try_to_timestamp(`{date_col}`, 'dd-MM-yyyy')").cast("date"),
        F.expr(f"try_to_timestamp(`{date_col}`, 'dd/MM/yyyy')").cast("date"),
        F.expr(f"try_to_timestamp(`{date_col}`, 'MM/dd/yyyy')").cast("date"),
        F.expr(f"try_to_timestamp(`{date_col}`, 'MM-dd-yyyy')").cast("date"),
        F.expr(f"try_to_timestamp(`{date_col}`, 'yyyy/MM/dd')").cast("date"),
        F.expr(f"try_to_timestamp(`{date_col}`, 'dd MMM yyyy')").cast("date"),
        F.expr(f"try_to_timestamp(`{date_col}`, 'MMM dd, yyyy')").cast("date")
    )

    silver_df = silver_df.withColumn(date_col, parsed_date)

print("Date standardization completed.")
silver_df.show(5, truncate=False)


Date standardization completed.
+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|customer_name     |product               |category |quantity|unit_price|discount_percent|payment_method  |city      |state      |order_date|order_status|
+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports   |2.0     |670.0     |30.0            |Credit Card     |Bengaluru |Karnataka  |2025-06-06|Cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books    |4.0     |670.0     |0.0             |Net Banking     |Pune      |Maharashtra|2025-05-08|Delivered   |
|100506  |C0228      |Diya Tripathi     |Competitive Exam Guide|Books    |6.0     |610.0

In [29]:
text_cols = [
    f.name for f in silver_df.schema.fields
    if isinstance(f.dataType, T.StringType)
]

if text_cols:
    silver_df = silver_df.fillna("Unknown", subset=text_cols)

fill_map = {}
if quantity_col:
    fill_map[quantity_col] = 0.0
if discount_col:
    fill_map[discount_col] = 0.0

if fill_map:
    silver_df = silver_df.fillna(fill_map)

print("Missing-value handling completed.")


Missing-value handling completed.


## 4. Verification

In [30]:
print("Original Bronze row count:", df.count())
print("Silver row count:", silver_df.count())
print("Rows removed during preprocessing:", df.count() - silver_df.count())

print("\nSilver schema:")
silver_df.printSchema()

print("\nRemaining null counts:")
silver_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in silver_df.columns
]).show(truncate=False)

if quantity_col:
    print("Invalid quantity values remaining:",
          silver_df.filter(F.col(quantity_col) < 0).count())

if unit_price_col:
    print("Invalid unit price values remaining:",
          silver_df.filter(F.col(unit_price_col) < 0).count())

if discount_col:
    print("Invalid discount values remaining:",
          silver_df.filter(
              (F.col(discount_col) < 0) | (F.col(discount_col) > 100)
          ).count())

print("\nSample Silver rows:")
silver_df.show(5, truncate=False)


Original Bronze row count: 1000
Silver row count: 990
Rows removed during preprocessing: 10

Silver schema:
root
 |-- order_id: integer (nullable = false)
 |-- customer_id: string (nullable = false)
 |-- customer_name: string (nullable = false)
 |-- product: string (nullable = false)
 |-- category: string (nullable = false)
 |-- quantity: double (nullable = false)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: double (nullable = false)
 |-- payment_method: string (nullable = false)
 |-- city: string (nullable = false)
 |-- state: string (nullable = false)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = false)


Remaining null counts:
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+------

In [31]:
silver_output_dir = os.path.join(BASE, "silver")
temp_output = os.path.join(BASE, "_silver_temp")

if os.path.exists(temp_output):
    shutil.rmtree(temp_output)

final_csv = os.path.join(silver_output_dir, "sales_silver.csv")
if os.path.exists(final_csv):
    os.remove(final_csv)

(
    silver_df.coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(temp_output)
)

part_files = glob.glob(os.path.join(temp_output, "part-*.csv"))
if not part_files:
    raise FileNotFoundError("Spark did not create the expected CSV part file.")

shutil.copy2(part_files[0], final_csv)
shutil.rmtree(temp_output)

print("Silver file created:", final_csv)
print("File size:", os.path.getsize(final_csv), "bytes")


Silver file created: /content/data_lake/silver/sales_silver.csv
File size: 113452 bytes


### Silver Data Verification

In [32]:
silver_check = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(final_csv)
)

print("Silver file read successfully.")
print("Silver row count:", silver_check.count())
print("\nSilver schema:")
silver_check.printSchema()

print("\nA few Silver rows:")
silver_check.show(5, truncate=False)


Silver file read successfully.
Silver row count: 990

Silver schema:
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)


A few Silver rows:
+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|customer_name     |product               |category |quantity|unit_price|discount_percent|payment_method  |city      |state      |order_date|order_status|
+--------+--

In [33]:
from google.colab import files
files.download(final_csv)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>